# 14: Entity Resolution for Banking Compliance

This notebook demonstrates multi-level entity resolution across three complexity tiers.

## Key Features
- **Standard Resolution**: Customer deduplication during onboarding
- **High-Complexity Resolution**: UBO/Sanctions multi-hop resolution
- **Ultra-High Resolution**: Synthetic identity fraud ring detection

## Use Cases
- Preventing duplicate customer records
- KYC/AML identity verification
- UBO identification through corporate structures
- Sanctions evasion detection
- Synthetic identity fraud detection

## Complexity Levels
| Level | Use Case | Signals | Accuracy |
|-------|----------|---------|----------|
| Standard | Deduplication | SSN, DOB, Name | 95-98% |
| High | UBO/Sanctions | Multi-hop, Cross-jurisdiction | 85-92% |
| Ultra-High | Fraud Rings | Network, Behavioral, Device | 75-85% |

**Author**: David LECONTE - IBM Worldwide | Data & AI | Tiger Team  
**Date**: 2026-03-23

In [ ]:
# Import required libraries
import sys
from pathlib import Path
from datetime import datetime
import hashlib
import json

# Add project root to path
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root))

# Fix asyncio event loop conflict in Jupyter
import nest_asyncio
nest_asyncio.apply()


import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from src.python.client.janusgraph_client import JanusGraphClient
from banking.analytics.entity_resolution import (
    EntityResolver,
    NameMatcher,
    ResolutionComplexity,
    MatchType,
    create_entity_resolution_report
)

print("✅ Libraries imported successfully")


## 1. Connect to JanusGraph

In [ ]:
# Connect to JanusGraph using auto-detected config
from notebook_config import JANUSGRAPH_CONFIG
from urllib.parse import urlparse

# Parse the URL to get host/port
parsed = urlparse(JANUSGRAPH_CONFIG['url'])
host = parsed.hostname
port = parsed.port

client = JanusGraphClient(
    host=host,
    port=port,
    use_ssl=JANUSGRAPH_CONFIG['use_ssl'],
    verify_certs=False
)
client.connect()

# Verify connection
vertex_count = client.execute("g.V().count()")[0]
edge_count = client.execute("g.E().count()")[0]

print(f"Connected to JanusGraph at {host}:{port}")
print(f"Total vertices: {vertex_count}")
print(f"Total edges: {edge_count}")


## 2. Generate and Load Entity Resolution Data

In [ ]:
# Generate deterministic entity resolution data
from banking.data_generators.scripts.generate_entity_resolution_data import (
    generate_entity_resolution_data_deterministic,
    load_into_janusgraph
)

# Generate data with seed=42 for determinism
print("Generating entity resolution data...")
er_data = generate_entity_resolution_data_deterministic(seed=42)

print(f"\nGenerated:")
print(f"  Vertices: {er_data['checksums']['total_vertices']}")
print(f"  Edges: {er_data['checksums']['total_edges']}")
print(f"  Persons: {er_data['checksums']['total_persons']}")
print(f"  Companies: {er_data['checksums']['total_companies']}")
print(f"  Addresses: {er_data['checksums']['total_addresses']}")
print(f"  Vertex checksum: {er_data['checksums']['vertices']}")
print(f"  Edge checksum: {er_data['checksums']['edges']}")

In [ ]:
# Display scenarios
print("\nScenarios Generated:")
print("="*60)
for name, scenario in er_data['scenarios'].items():
    print(f"\n{name.upper().replace('_', ' ')}:")
    print(f"  Description: {scenario['description']}")
    print(f"  Complexity: {scenario['complexity']}")
    print(f"  Entity count: {scenario['entity_count']}")
    print(f"  Expected: {scenario['expected_resolution']}")

In [ ]:
# Load data into JanusGraph
print("Loading entity resolution data into JanusGraph...")
counts = load_into_janusgraph(er_data)
print(f"✅ Created {counts['vertices']} vertices and {counts['edges']} edges")

## 3. Initialize Entity Resolver

In [ ]:
# Initialize entity resolver
resolver = EntityResolver(client)

print("✅ Entity resolver initialized")
print("\nAttribute Weights:")
print("\nStandard Resolution:")
for attr, weight in EntityResolver.ATTRIBUTE_WEIGHTS['standard'].items():
    print(f"  {attr}: {weight:.2f}")

print("\nHigh-Complexity Resolution:")
for attr, weight in EntityResolver.ATTRIBUTE_WEIGHTS['high_complexity'].items():
    print(f"  {attr}: {weight:.2f}")

print("\nUltra-High Resolution:")
for attr, weight in EntityResolver.ATTRIBUTE_WEIGHTS['ultra_high'].items():
    print(f"  {attr}: {weight:.2f}")

---
## SCENARIO 1: Standard Customer Deduplication

### Business Context
A customer attempts to open a new account. We need to determine if this person already exists to:
- Prevent duplicate records
- Apply existing KYC status
- Avoid redundant onboarding costs

In [ ]:
# Get standard scenario entities
standard_entities = er_data['vertices'][:3]  # First 3 are standard duplicates

print("STANDARD SCENARIO: Customer Duplicates")
print("="*60)
print("\nOriginal Customer:")
original = standard_entities[0]
print(f"  ID: {original['id'][:16]}...")
print(f"  Name: {original['properties']['name']}")
print(f"  SSN: {original['properties']['ssn']}")
print(f"  DOB: {original['properties']['dateOfBirth']}")
print(f"  Phone: {original['properties']['phone']}")
print(f"  Email: {original['properties']['email']}")

print("\nDuplicate 1 (Name Variant):")
dup1 = standard_entities[1]
print(f"  ID: {dup1['id'][:16]}...")
print(f"  Name: {dup1['properties']['name']}")
print(f"  SSN: {dup1['properties']['ssn']}")
print(f"  DOB: {dup1['properties']['dateOfBirth']}")
print(f"  Phone: {dup1['properties']['phone']}")
print(f"  Email: {dup1['properties']['email']}")

print("\nDuplicate 2 (Nickname):")
dup2 = standard_entities[2]
print(f"  ID: {dup2['id'][:16]}...")
print(f"  Name: {dup2['properties']['name']} (Jack is nickname for John)")
print(f"  SSN: {dup2['properties']['ssn']}")
print(f"  DOB: {dup2['properties']['dateOfBirth']}")
print(f"  Phone: {dup2['properties']['phone']}")
print(f"  Email: {dup2['properties']['email']}")

In [ ]:
# Demonstrate name matching
print("\nNAME MATCHING DEMONSTRATION")
print("="*60)

name_pairs = [
    ("John A. Smith", "Jonathan Smith"),
    ("John Smith", "Jack Smith"),
    ("John", "Ivan"),
    ("Smith", "Schmidt"),
    ("Robert", "Bob"),
    ("William", "Bill"),
]

for name_a, name_b in name_pairs:
    score, match_type, explanation = NameMatcher.compare_names(name_a, name_b)
    print(f"\n'{name_a}' vs '{name_b}':")
    print(f"  Score: {score:.2f}")
    print(f"  Type: {match_type.value}")
    print(f"  Explanation: {explanation}")

In [ ]:
# Resolve standard duplicates
print("\nRESOLUTION RESULTS")
print("="*60)

def classify_ambiguity(confidence):
    if confidence >= 0.90:
        return "LOW_AMBIGUITY"
    if confidence >= 0.70:
        return "MEDIUM_AMBIGUITY"
    return "HIGH_AMBIGUITY"

def render_merge_rationale_card(label, match):
    decision = "MERGE" if match.confidence >= 0.90 else "REVIEW"
    top_signals = sorted(
        match.match_signals,
        key=lambda s: (s.score * s.weight),
        reverse=True,
    )[:3]
    signal_summary = "; ".join(
        f"{signal.attribute}: {signal.score:.2f} x {signal.weight:.2f}"
        for signal in top_signals
    )
    print(f"\n📌 {label} — MERGE RATIONALE CARD")
    print(f"  Decision: {decision}")
    print(f"  Confidence: {match.confidence:.2%}")
    print(f"  Ambiguity Class: {classify_ambiguity(match.confidence)}")
    print(f"  Engine Action: {match.resolution_action.upper()}")
    print(f"  Top Signals: {signal_summary}")
    return top_signals, signal_summary

def build_resolution_evidence(label, match, top_signals, signal_summary):
    ambiguity_class = classify_ambiguity(match.confidence)
    alert_seed = f"{label}|{match.entity_a_id}|{match.entity_b_id}|{match.confidence:.6f}|{ambiguity_class}"
    alert_id = f"ER-{hashlib.sha256(alert_seed.encode('utf-8')).hexdigest()[:12]}"
    timestamp_utc = datetime(2026, 1, 15, 12, 0, 0).isoformat() + "Z"
    decision = "MERGE" if match.confidence >= 0.90 else "REVIEW"
    return {
        "alert_id": alert_id,
        "timestamp": timestamp_utc,
        "detector": "EntityResolver",
        "decision": decision,
        "confidence": round(match.confidence * 100, 2),
        "risk_score": round(1.0 - match.confidence, 4),
        "victim_id": match.entity_a_id,
        "counterparty_id": match.entity_b_id,
        "complexity": match.complexity.value,
        "resolution_action": match.resolution_action,
        "ambiguity_class": ambiguity_class,
        "reason_codes": [f"ER_{signal.attribute.upper()}" for signal in top_signals],
        "evidence_summary": f"{label}: confidence={match.confidence:.2%}, top_signals={signal_summary}",
    }

# Resolve original vs duplicate 1
match1 = resolver.resolve(
    original['id'],
    dup1['id'],
    ResolutionComplexity.STANDARD
)
top_signals_1, signal_summary_1 = render_merge_rationale_card("Original vs Duplicate 1", match1)
print("  Signals:")
for signal in match1.match_signals:
    print(f"    - {signal.attribute}: {signal.score:.2f} (weight: {signal.weight:.2f})")
    print(f"      {signal.explanation}")

# Resolve original vs duplicate 2
match2 = resolver.resolve(
    original['id'],
    dup2['id'],
    ResolutionComplexity.STANDARD
)
top_signals_2, signal_summary_2 = render_merge_rationale_card("Original vs Duplicate 2", match2)
print("  Signals:")
for signal in match2.match_signals:
    print(f"    - {signal.attribute}: {signal.score:.2f} (weight: {signal.weight:.2f})")
    print(f"      {signal.explanation}")

# Export deterministic case evidence with ambiguity labels
evidence_records = [
    build_resolution_evidence("Original vs Duplicate 1", match1, top_signals_1, signal_summary_1),
    build_resolution_evidence("Original vs Duplicate 2", match2, top_signals_2, signal_summary_2),
]
export_dir = Path("exports/evidence/entity_resolution")
export_dir.mkdir(parents=True, exist_ok=True)
for record in evidence_records:
    evidence_path = export_dir / f"{record['alert_id']}_evidence.json"
    evidence_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
summary_path = export_dir / "entity_resolution_alerts_summary.csv"
pd.DataFrame(evidence_records).to_csv(summary_path, index=False)
print(f"\n✅ Entity-resolution evidence exported to: {export_dir}")
print(f"📊 Summary CSV: {summary_path}")

In [ ]:
# Business impact visualization
print("\nBUSINESS IMPACT - STANDARD RESOLUTION")
print("="*60)

impact_metrics = {
    'Duplicate Rate': {'Before': '8-12%', 'After': '<2%'},
    'Onboarding Cost': {'Before': '$150/customer', 'After': '$50/customer'},
    'KYC Re-verification': {'Before': '15%', 'After': '5%'},
    'Customer Complaints': {'Before': '120/month', 'After': '30/month'},
}

impact_df = pd.DataFrame(impact_metrics).T
print("\n" + impact_df.to_string())
print("\n✅ Result: All 3 records should be MERGED into single customer profile")

---
## SCENARIO 2: High-Complexity UBO/Sanctions Resolution

### Business Context
An investigator needs to determine if multiple companies across different jurisdictions are controlled by the same person - and whether that person appears on a sanctions list.

### Complexity Factors
- Multiple corporate entities across jurisdictions (UK, BVI, Cyprus, Russia)
- Trust structures obscure ownership
- Name variants in different languages
- 3-5 hops through ownership chains
- Sanctions screening with fuzzy matching

In [ ]:
# Identify UBO scenario entities
ubo_persons = [v for v in er_data['vertices'] 
               if v['label'] == 'Person' and v['properties'].get('isUBO')]
ubo_companies = [v for v in er_data['vertices'] 
                 if v['label'] == 'Company' and v['properties'].get('jurisdiction')]

print("HIGH-COMPLEXITY SCENARIO: UBO/Sanctions Evasion")
print("="*60)
print("\nUBO Identities (Same Person, Different Passports):")
print("-"*60)

for person in ubo_persons:
    props = person['properties']
    sanctioned = props.get('sanctioned', False)
    flag = '⚠️ SANCTIONED' if sanctioned else ''
    print(f"\n{props['nationality']}: {props['name']} {flag}")
    print(f"  Passport: {props.get('passport', 'N/A')}")
    print(f"  DOB: {props['dateOfBirth']}")
    print(f"  Risk Score: {props.get('riskScore', 0):.2f}")
    if sanctioned:
        print(f"  Sanctions List: {props.get('sanctionsList', 'Unknown')}")

In [ ]:
# Display corporate structure
print("\nCORPORATE STRUCTURE")
print("="*60)

for company in ubo_companies:
    props = company['properties']
    print(f"\n{props['name']} ({props.get('jurisdiction', 'N/A')})")
    print(f"  ID: {company['id'][:16]}...")
    if props.get('role'):
        print(f"  Role: {props['role']}")

# Trust
trust = [v for v in er_data['vertices'] if v['label'] == 'Trust']
if trust:
    print(f"\n{trust[0]['properties']['name']} ({trust[0]['properties']['jurisdiction']})")
    print(f"  Role: Trust vehicle for ownership obfuscation")

In [ ]:
# Resolve UBO identities using HIGH complexity
print("\nUBO RESOLUTION RESULTS")
print("="*60)

# Compare identities pairwise
print("\nCross-Jurisdictional Identity Resolution:")
for i in range(len(ubo_persons) - 1):
    for j in range(i + 1, len(ubo_persons)):
        match = resolver.resolve(
            ubo_persons[i]['id'],
            ubo_persons[j]['id'],
            ResolutionComplexity.HIGH
        )
        name_a = ubo_persons[i]['properties']['name']
        name_b = ubo_persons[j]['properties']['name']
        nat_a = ubo_persons[i]['properties']['nationality']
        nat_b = ubo_persons[j]['properties']['nationality']
        print(f"\n  {nat_a}:{name_a} vs {nat_b}:{name_b}")
        print(f"    Confidence: {match.confidence:.2f}")
        print(f"    Action: {match.resolution_action.upper()}")

In [ ]:
# Cultural name matching demonstration
print("\nCULTURAL NAME EQUIVALENTS")
print("="*60)

cultural_pairs = [
    ("John", "Ivan", "English/Russian"),
    ("John", "Hans", "English/German"),
    ("John", "Jean", "English/French"),
    ("John", "Juan", "English/Spanish"),
    ("Smith", "Schmidt", "English/German translation"),
]

for name_a, name_b, context in cultural_pairs:
    score, match_type, explanation = NameMatcher.compare_names(name_a, name_b)
    print(f"\n'{name_a}' ≈ '{name_b}' ({context})")
    print(f"  Score: {score:.2f} | Type: {match_type.value}")

In [ ]:
# Regulatory requirements met
print("\nREGULATORY REQUIREMENTS MET")
print("="*60)

regulations = [
    ("FATF Rec. 24/25", "UBO identification", "Multi-hop ownership traversal"),
    ("EU AMLD6", "25% threshold + control", "Control relationship inference"),
    ("UK PSC Register", "Significant control", "Direct + indirect ownership"),
    ("US CTA", "Beneficial ownership", "Corporate structure resolution"),
    ("OFAC", "Sanctions screening", "Fuzzy matching + network analysis"),
]

print("\n{:<20} {:<30} {:<30}".format('Regulation', 'Requirement', 'How Addressed'))
print("-"*80)
for reg, req, how in regulations:
    print("{:<20} {:<30} {:<30}".format(reg, req, how))

print("\n⚠️ ALERT: One identity matches OFAC SDN List - Immediate action required")

---
## SCENARIO 3: Ultra-High Synthetic Identity Fraud Detection

### Business Context
A fraud ring has created synthetic identities by combining:
- Real SSNs (stolen from children, elderly, or deceased)
- Fabricated names and addresses
- Artificial credit histories

These identities build credit over 6-12 months, then "bust out" with maximum loans.

### Why Ultra-High Complexity?
- Each identity appears unique on paper
- SSNs are valid (stolen from real people)
- Gradual credit building avoids alerts
- Detection requires network + behavioral + device analysis

In [ ]:
# Identify synthetic identities
synth_persons = [v for v in er_data['vertices'] 
                 if v['label'] == 'Person' and v['properties'].get('synthetic')]
address = [v for v in er_data['vertices'] if v['label'] == 'Address'][0]
device = [v for v in er_data['vertices'] if v['label'] == 'Device'][0]

print("ULTRA-HIGH SCENARIO: Synthetic Identity Fraud Ring")
print("="*60)
print(f"\nFraud Ring Size: {len(synth_persons)} synthetic identities")
print(f"Shared Address: {address['properties']['street']}, {address['properties']['city']}")
print(f"Shared Device Fingerprint: {device['properties']['fingerprint']}")

print("\nSynthetic Identities (showing first 5):")
print("-"*60)
for i, person in enumerate(synth_persons[:5]):
    props = person['properties']
    print(f"\n{i+1}. {props['name']}")
    print(f"   SSN: {props['ssn']} (stolen)")
    print(f"   DOB: {props['dateOfBirth']} (fabricated)")
    print(f"   Phone: {props['phone']} (sequential)")

In [ ]:
# Analyze fraud ring patterns
print("\nFRAUD RING PATTERN ANALYSIS")
print("="*60)

print("\nSequential SSN Pattern (stolen from same source):")
ssns = [p['properties']['ssn'] for p in synth_persons]
for i, ssn in enumerate(ssns[:5]):
    print(f"  {i+1}. {ssn}")
print(f"  ... ({len(ssns)} total)")

print("\nSequential DOB Pattern (fabricated):")
dobs = [p['properties']['dateOfBirth'] for p in synth_persons]
for i, dob in enumerate(dobs[:5]):
    print(f"  {i+1}. {dob}")

print("\nSequential Phone Pattern:")
phones = [p['properties']['phone'] for p in synth_persons]
for i, phone in enumerate(phones[:5]):
    print(f"  {i+1}. {phone}")

In [ ]:
# Network visualization of fraud ring
print("\nFRAUD RING NETWORK STRUCTURE")
print("="*60)

print("\nNetwork Indicators:")
print("  • All identities registered at SAME ADDRESS")
print("  • All identities used SAME DEVICE")
print("  • Sequential SSNs (stolen in batch)")
print("  • Sequential DOBs (fabricated pattern)")
print("  • Sequential phone numbers")
print("  • Same last name pattern (Johnson)")

print("\nDetection Signals:")
signals = [
    ("Address Cluster", 0.30, "47 identities at single address"),
    ("Device Correlation", 0.25, "All used same device fingerprint"),
    ("Sequential SSN", 0.20, "SSNs sequential (stolen batch)"),
    ("Temporal Clustering", 0.15, "All created in same period"),
    ("Behavioral Similarity", 0.10, "Identical application patterns"),
]

total_signal = 0
for signal, weight, desc in signals:
    print(f"  {signal}: {weight:.2f} - {desc}")
    total_signal += weight

print(f"\n  TOTAL CONFIDENCE: {total_signal:.2f}")

In [ ]:
# Business impact
print("\nBUSINESS IMPACT - SYNTHETIC IDENTITY DETECTION")
print("="*60)

print("\nEstimated Exposure:")
avg_credit = 15000  # Average credit per synthetic identity
total_exposure = len(synth_persons) * avg_credit
print(f"  Synthetic Identities: {len(synth_persons)}")
print(f"  Average Credit Limit: ${avg_credit:,}")
print(f"  Total Exposure: ${total_exposure:,}")

print("\nDetection Value:")
print(f"  Fraud Prevented: ${total_exposure:,}")
print(f"  Investigation Time Saved: 85% (network-based detection)")
print(f"  False Positive Reduction: 60% (multi-signal approach)")

print("\n✅ Result: All synthetic identities detected as FRAUD RING")
print("   Action: Freeze accounts, investigate, file SAR")

---
## Summary and Best Practices

In [ ]:
# Summary comparison
print("ENTITY RESOLUTION COMPLEXITY SUMMARY")
print("="*60)

summary = {
    'Standard': {
        'Use Case': 'Customer Deduplication',
        'Signals': 'SSN, DOB, Name, Phone',
        'Hops': '1-2',
        'Accuracy': '95-98%',
        'Business Value': 'Operational Efficiency',
    },
    'High': {
        'Use Case': 'UBO/Sanctions',
        'Signals': 'Multi-hop, Cross-jurisdiction',
        'Hops': '3-5',
        'Accuracy': '85-92%',
        'Business Value': 'Compliance, Risk',
    },
    'Ultra-High': {
        'Use Case': 'Fraud Rings',
        'Signals': 'Network, Behavioral, Device',
        'Hops': '5+',
        'Accuracy': '75-85%',
        'Business Value': 'Loss Prevention',
    }
}

summary_df = pd.DataFrame(summary).T
print("\n" + summary_df.to_string())

In [ ]:
# Key takeaways
print("\n" + "="*60)
print("KEY TAKEAWAYS")
print("="*60)

takeaways = [
    "1. Graph-based resolution incorporates relationship signals",
    "   that traditional pairwise comparison misses",
    "",
    "2. Name matching must handle nicknames, cultural variants,",
    "   transliteration, and fuzzy matches",
    "",
    "3. High-complexity resolution requires multi-hop traversal",
    "   through corporate structures and trust arrangements",
    "",
    "4. Ultra-high complexity detection combines network analysis,",
    "   behavioral patterns, and device intelligence",
    "",
    "5. Confidence thresholds must be calibrated by use case:",
    "   - 0.95+ for auto-merge",
    "   - 0.85-0.95 for auto-link",
    "   - 0.70-0.85 for review",
    "   - <0.70 for no match",
]

for line in takeaways:
    print(line)

In [ ]:
# Close connection
client.close()
print("\n✅ Demo complete - Connection closed")